# Dataspace PUSH Transfer Flow

This notebook demonstrates the execution of the Dataspace Protocol (DSP 2025-1) transfer process in PUSH mode over Eunomia RPC (`/dsp/current/transfers/rpc/setup-*`):
1. Executes prerequisites and contract negotiation (`contract_negotiation.ipynb` with `setup_catalog_policies.ipynb`).
2. Initiates PUSH transfer request specifying a destination sink `dataAddress` (`setup-request`).
3. Starts push transfer from Provider (`setup-start`).
4. Suspends transfer from Provider (`setup-suspension`).
5. Resumes transfer from Provider (`setup-start`).
6. Enforces protocol role constraint (rejecting Consumer-initiated start).
7. Completes push transfer from Provider (`setup-completion`).
8. Queries final transfer process state via the Transfer Agent Management API.

## 1. Prerequisites
Execute catalog and policies setup, and complete DSP 2025-1 Contract Negotiation to obtain an active contract agreement ID.

In [ ]:
%run ./setup_catalog_policies.ipynb
%run ./contract_negotiation.ipynb

## 2. Transfer Request Setup with Sink DataAddress (Consumer -> Provider)
The Consumer initiates the PUSH transfer by calling `/dsp/current/transfers/rpc/setup-request`, specifying the format `http+pushing` and providing its destination sink `dataAddress` where data should be delivered.
The Provider accepts the request, creates the process in `REQUESTED` state, and returns `providerPid`.

In [1]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-request"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+pushing",
    "dataAddress": {
        "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
        "endpoint": "http://127.0.0.1:1112",
        "endpointProperties": [
            {
                "name": "authorization",
                "value": "TOKEN-ABCDEFG"
            },
            {
                "name": "authType",
                "value": "bearer"
            }
        ]
    },
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
transfer_process_consumer_pid = response_as_json["transferAgentModel"]["correlation"]["consumerPid"]
transfer_process_provider_pid = response_as_json["transferAgentModel"]["correlation"]["providerPid"]
print(json.dumps(response_as_json, indent=2))


NameError: name 'data_space_consumer' is not defined

## 3. Start Push Transfer (Provider -> Consumer)
The Provider invokes `/dsp/current/transfers/rpc/setup-start`. The Provider's dataplane begins pushing data to the Consumer's designated sink.
The transfer process moves from `REQUESTED` to `STARTED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 4. Suspend Transfer (Provider -> Consumer)
The Provider temporarily suspends the active data transmission by calling `/dsp/current/transfers/rpc/setup-suspension`.
The process transitions from `STARTED` to `SUSPENDED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-suspension"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferSuspensionMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"],
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 5. Resume Push Transfer (Provider -> Consumer)
The Provider resumes the suspended push transfer by sending a `RpcTransferStartMessage` to `/dsp/current/transfers/rpc/setup-start`.
The transfer process returns to `STARTED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 6. Role Validation Test: Reject Consumer-Initiated Start
Verify DSP role constraint enforcement: In DSP 2025-1, only a Provider may send a `TransferStartMessage`.
Attempting to invoke `/dsp/current/transfers/rpc/setup-start` as the Consumer fails with a role validation error.

In [ ]:
url = data_space_consumer + "/dsp/current/transfers/rpc/setup-start"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
print("Status Code:", response.status_code)
try:
    print(json.dumps(response.json(), indent=2))
except Exception:
    print(response.text)


## 7. Complete Transfer (Provider -> Consumer)
Once the Provider has delivered all data items to the Consumer's sink, the Provider signals completion via `/dsp/current/transfers/rpc/setup-completion`.
A `TransferCompletionMessage` is sent over DSP, moving the transfer process to the terminal state `COMPLETED`.

In [ ]:
url = data_space_provider + "/dsp/current/transfers/rpc/setup-completion"
payload = {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferCompletionMessage",
    "consumerPid": transfer_process_consumer_pid,
    "providerPid": transfer_process_provider_pid,
}
response = requests.request("POST", url, headers=json_header, data=json.dumps(payload))
response_as_json = response.json()
print(json.dumps(response_as_json, indent=2))


## 8. Query Final Transfer Process State
Query the Provider's Transfer Agent Management API (`/api/v1/transfer-agent/transfer-processes`) to inspect the final recorded process state.

In [ ]:
url = data_space_provider + "/api/v1/transfer-agent/transfer-processes"
response = requests.request("GET", url, headers=json_header)
print("Processes in Provider:")
print(json.dumps(response.json(), indent=2))
